# Xem nhanh cấu trúc dữ liệu MBD-mini

Notebook hiển thị **5 dòng đầu của từng bảng**, kèm tên cột và kiểu dữ liệu Parquet/Arrow.
Dữ liệu đã giải nén nằm tại `data/MBD-mini/`.

**Cách chạy:** chọn kernel Python trong `.venv`, sau đó **Run All**.
Nếu thiếu thư viện, chạy trong terminal ở thư mục gốc repo:

```powershell
.\.venv\Scripts\python.exe -m pip install pandas pyarrow ipykernel
```

Mẫu được lấy theo thứ tự đường dẫn file Parquet, không sắp xếp theo thời gian.
Cột `fold` được đọc từ các thư mục `fold=0`, ..., `fold=4`.
Các mảng dài được rút gọn khi hiển thị; DataFrame mẫu vẫn giữ nguyên giá trị.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import Markdown, display

# Tìm repo khi chạy từ thư mục gốc, notebooks/ hoặc thư mục con.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file() and (path / "data" / "MBD-mini").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy data/MBD-mini/. Hãy chạy notebook trong repo này.")

DATA_DIR = PROJECT_ROOT / "data" / "MBD-mini"
N_ROWS = 5
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_seq_items", 6)
np.set_printoptions(threshold=6, edgeitems=2)
print(f"Thư mục dữ liệu: {DATA_DIR}")


## Các bảng dữ liệu

`detail` lưu từng sự kiện; `ptls` gom chuỗi sự kiện của khách hàng thành các mảng.
`client_split` chứa cách chia fold, còn `targets` chứa nhãn và thông tin theo tháng.


In [ ]:
TABLES = {
    "client_split": "Khách hàng và fold dùng cho cross-validation",
    "targets": "Nhãn sản phẩm và thông tin theo tháng",
    "detail/dialog": "Từng lượt hội thoại và embedding",
    "detail/geo": "Từng sự kiện vị trí",
    "detail/trx": "Từng giao dịch",
    "ptls/dialog": "Chuỗi hội thoại của khách hàng",
    "ptls/geo": "Chuỗi sự kiện vị trí của khách hàng",
    "ptls/trx": "Chuỗi giao dịch của khách hàng",
}
parquet_files = {
    name: sorted((DATA_DIR / name).rglob("*.parquet"))
    for name in TABLES
}
missing = [name for name, files in parquet_files.items() if not files]
if missing:
    raise FileNotFoundError(f"Không tìm thấy file Parquet trong: {missing}")

display(pd.DataFrame([
    {"Bảng": name, "Mô tả": description, "Số file Parquet": len(parquet_files[name])}
    for name, description in TABLES.items()
]))


## Hàm xem mẫu

Đọc theo batch từ các file Parquet theo thứ tự đường dẫn và dừng ngay khi đủ 5 dòng.
Cột `fold` được bổ sung từ đường dẫn partition. Bảng schema phân biệt rõ các cột số,
thời gian và mảng lồng nhau.


In [ ]:
def preview_table(name):
    samples = []
    rows_read = 0
    for path in parquet_files[name]:
        with pq.ParquetFile(path) as parquet:
            if parquet.metadata.num_rows == 0:
                continue
            batch = next(parquet.iter_batches(batch_size=N_ROWS - rows_read))
            sample = pa.Table.from_batches([batch])

        # Spark lưu fold trong tên thư mục thay vì bên trong file Parquet.
        if "fold" not in sample.column_names:
            fold = next(int(part.split("=", 1)[1]) for part in path.parts if part.startswith("fold="))
            sample = sample.append_column(
                "fold", pa.array([fold] * sample.num_rows, type=pa.int32())
            )
        samples.append(sample)
        rows_read += sample.num_rows
        if rows_read >= N_ROWS:
            break

    if not samples:
        raise ValueError(f"Bảng {name} không có dòng dữ liệu.")
    sample = pa.concat_tables(samples)
    frame = sample.to_pandas()

    display(Markdown(f"**{len(frame)} dòng mẫu · {len(sample.schema)} cột**"))
    display(frame)
    display(Markdown("**Tên cột và kiểu dữ liệu**"))
    display(pd.DataFrame([
        {"Cột": field.name, "Kiểu Arrow": str(field.type), "Kiểu pandas": str(frame[field.name].dtype)}
        for field in sample.schema
    ]))
    return frame


## client_split

Khách hàng và fold dùng cho cross-validation.


In [ ]:
client_split = preview_table("client_split")


## targets

Nhãn sản phẩm và thông tin khách hàng theo tháng.


In [ ]:
targets = preview_table("targets")


## detail/dialog

Mỗi dòng là một sự kiện hội thoại, có embedding dạng mảng.


In [ ]:
detail_dialog = preview_table("detail/dialog")


## detail/geo

Mỗi dòng là một sự kiện vị trí.


In [ ]:
detail_geo = preview_table("detail/geo")


## detail/trx

Mỗi dòng là một giao dịch.


In [ ]:
detail_trx = preview_table("detail/trx")


## ptls/dialog

Dữ liệu hội thoại dạng chuỗi; một số cột là mảng hoặc mảng lồng nhau.


In [ ]:
ptls_dialog = preview_table("ptls/dialog")


## ptls/geo

Dữ liệu vị trí dạng chuỗi.


In [ ]:
ptls_geo = preview_table("ptls/geo")


## ptls/trx

Dữ liệu giao dịch dạng chuỗi.


In [ ]:
ptls_trx = preview_table("ptls/trx")
